In [1]:
import os
import io
import base64

import pymupdf
import numpy as np
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model


C:\Users\Adit Sharma\AppData\Local\Temp\ipykernel_8580\3752200945.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
import matplotlib.pyplot as plt

pdf_path = "multimodal_sample.pdf"

if not os.path.exists(pdf_path):
    # bar chart → PNG bytes
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(["Q1", "Q2", "Q3"], [120, 180, 310], color=["blue", "green", "red"])
    ax.set_title("Revenue by Quarter")
    ax.set_ylabel("Revenue ($K)")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120)
    plt.close(fig)

    # one page: text on top, chart below
    text = (
        "Annual Revenue Overview\n"
        "This document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\n"
        "below, revenue grew steadily with the highest growth recorded in Q3.\n"
        "Q1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\n"
        "Q1 due to marketing campaigns. Q3 had exponential growth due to global expansion."
    )
    pdf = pymupdf.open()
    page = pdf.new_page()
    page.insert_text((50, 60), text, fontsize=10)
    page.insert_image(pymupdf.Rect(80, 160, 480, 400), stream=buf.getvalue())
    pdf.save(pdf_path)
    pdf.close()


In [3]:
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("groq_api")

# CLIP: one model, two encoders (text + image) sharing a 512-d space
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

# vision-capable chat model on Groq
llm = init_chat_model("groq:qwen/qwen3.8-27b")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [4]:
def embed_image(image_data):
    """Embed an image (file path or PIL Image) with CLIP."""
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs).pooler_output   # transformers 5: take .pooler_output
    features = features / features.norm(dim=-1, keepdim=True)              # unit length → dot product = cosine
    return features.squeeze().numpy()


def embed_text(text):
    """Embed text with CLIP (max 77 tokens; longer text is cut off)."""
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs).pooler_output
    features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().numpy()


In [5]:
doc = pymupdf.open(pdf_path)

all_docs = []            # Documents (text chunks + image placeholders)
all_embeddings = []      # one CLIP vector per Document, same order
image_data_store = {}    # image_id → base64 PNG, sent to the LLM later

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

for i, page in enumerate(doc):
    # text
    text = page.get_text()
    if text.strip():
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        for chunk in splitter.split_documents([temp_doc]):
            all_embeddings.append(embed_text(chunk.page_content))
            all_docs.append(chunk)

    # images
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            image_bytes = doc.extract_image(xref)["image"]
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            image_id = f"page_{i}_img_{img_index}"

            # keep the real image for the vision LLM
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            image_data_store[image_id] = base64.b64encode(buffered.getvalue()).decode()

            # vector for retrieval
            all_embeddings.append(embed_image(pil_image))
            all_docs.append(Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id},
            ))
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")

doc.close()
all_docs


[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [6]:
embeddings_array = np.array(all_embeddings)
print(embeddings_array.shape)   # (2, 512) for the sample PDF

vector_store = FAISS.from_embeddings(
    text_embeddings=[(d.page_content, emb) for d, emb in zip(all_docs, embeddings_array)],
    embedding=None,   # vectors are already computed; we always search by vector
    metadatas=[d.metadata for d in all_docs],
)


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


(2, 512)


In [7]:
def retrieve_multimodal(query, k=5):
    """Embed the query with CLIP's text encoder and search text + images together."""
    query_embedding = embed_text(query)
    return vector_store.similarity_search_by_vector(embedding=query_embedding, k=k)


In [8]:
def create_multimodal_message(query, retrieved_docs):
    """One HumanMessage holding the question, text excerpts and base64 images."""
    content = [{"type": "text", "text": f"Question: {query}\n\nContext:\n"}]

    text_docs = [d for d in retrieved_docs if d.metadata.get("type") == "text"]
    image_docs = [d for d in retrieved_docs if d.metadata.get("type") == "image"]

    if text_docs:
        text_context = "\n\n".join(f"[Page {d.metadata['page']}]: {d.page_content}" for d in text_docs)
        content.append({"type": "text", "text": f"Text excerpts:\n{text_context}\n"})

    for d in image_docs:
        image_id = d.metadata.get("image_id")
        if image_id in image_data_store:
            content.append({"type": "text", "text": f"\n[Image from page {d.metadata['page']}]:\n"})
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{image_data_store[image_id]}"},
            })

    content.append({"type": "text", "text": "\n\nPlease answer the question based on the provided text and images."})
    return HumanMessage(content=content)


In [9]:
def multimodal_pdf_rag_pipeline(query):
    context_docs = retrieve_multimodal(query, k=5)
    message = create_multimodal_message(query, context_docs)
    response = llm.invoke([message])

    print(f"\nRetrieved {len(context_docs)} documents:")
    for d in context_docs:
        if d.metadata.get("type") == "text":
            print(f"  - Text from page {d.metadata['page']}: {d.page_content[:100]}...")
        else:
            print(f"  - Image from page {d.metadata['page']}")
    print()
    return response.content


In [10]:
queries = [
    "What does the chart on page 1 show about revenue trends?",
    "Summarize the main findings from the document",
    "What visual elements are present in the document?",
]

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 50)
    print(f"Answer: {multimodal_pdf_rag_pipeline(query)}")
    print("=" * 70)



Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 0

Answer: Based on the chart and text provided, the revenue trends are as follows:

*   **Steady Growth:** Revenue increased consistently across the three quarters.
*   **Specific Values:**
    *   **Q1:** Approximately $120,000 (blue bar).
    *   **Q2:** Approximately $180,000 (green bar).
    *   **Q3:** Approximately $310,000 (red bar).
*   **Highest Growth in Q3:** The text notes that the highest growth was recorded in Q3, which is visually supported by the red bar being significantly taller than the others.
*   **Reasons for Growth (from text):**
    *   **Q1:** Moderate increase due to new product lines.
    *   **Q2:** Outperformed Q1 due to marketing campaigns.
    *   **Q3:** Experienced expone